<a href="https://colab.research.google.com/github/25wh1a0529/Nasscom-lab/blob/main/U8_Data_Cleaning_Part2_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)


# -----------------------------------------------------------
# A SMALL, ALREADY-CLEAN DATASET (Part 1 did the cleaning)
# -----------------------------------------------------------
# Mixed columns: an unordered category (city), an ordered category
# (size), two numeric features on very different scales, and a target.
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})
df



Setup complete. pandas 2.2.3


,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


In [13]:
#1. Encoding categorical data

# -----------------------------------------------------------
# 🔹 1A. ONE-HOT ENCODING (for UNORDERED categories)
# -----------------------------------------------------------

# 'city' has no natural order -> one 0/1 column per category
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe


# -----------------------------------------------------------
# 🔹 1B. LABEL / ORDINAL ENCODING (for ORDERED categories)
# -----------------------------------------------------------

# 'size' has a real order: small < medium < large -> map to 0,1,2
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]

,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


In [3]:
#2. Feature scaling
# 🔹 2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])


# 🔹 2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num = df[['age', 'income']]

z = StandardScaler().fit_transform(num)
m = MinMaxScaler().fit_transform(num)

print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))

        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0
Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


In [19]:

from sklearn.preprocessing import StandardScaler, MinMaxScaler

income = df[['income']]   # 2D shape for sklearn

# 1. standardise -> check mean ~0, std ~1
scaler = StandardScaler()
income_standard = scaler.fit_transform(income)

print("Mean:", income_standard.mean())
print("Std:", income_standard.std())

# 2. min-max scale -> check min 0, max 1
minmax = MinMaxScaler()
income_minmax = minmax.fit_transform(income)

print("Min:", income_minmax.min())
print("Max:", income_minmax.max())

# 3. Needs scaling: K-Nearest Neighbors (KNN).
# Doesn't need it: Decision Tree.

Mean: -2.7755575615628914e-17
Std: 1.0
Min: 0.0
Max: 1.0000000000000002


In [4]:
#3. Avoiding data leakage

# 🔹 3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))


train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


In [23]:
# 1a. WRONG: fit on ALL data, then split (leakage!)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
wrong_mean = StandardScaler().fit(X).mean_

# 1b. RIGHT: fit on TRAIN only
right_mean = StandardScaler().fit(X_train).mean_

print('fit-on-all mean :', wrong_mean.round(1))
print('fit-on-train mean:', right_mean.round(1))

# 2. Why fitting on all data is a problem:
# Fitting on all data causes data leakage because the scaler learns
# information (such as the mean and standard deviation) from the test set.
# This makes the evaluation less reliable because the test data has
# influenced preprocessing before the model is evaluated.

ValueError: could not convert string to float: 'pune'

In [5]:
#4. Feature engineering
# 🔹 4A. COMBINE & BIN EXISTING COLUMNS

fe = df.copy()
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)

fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                         labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]


# 🔹 4B. EXTRACT FROM A DATE

dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d

,date,day_of_week,month,is_weekend
0,2024-01-06,Saturday,1,True
1,2024-03-15,Friday,3,False
2,2024-07-21,Sunday,7,True
3,2024-12-25,Wednesday,12,False


In [24]:
ex = df.copy()

# 1. high_earner flag (income > median)
median_income = ex['income'].median()
ex['high_earner'] = ex['income'] > median_income

# 2. bin income into 3 buckets with pd.cut
ex['income_bucket'] = pd.cut(
    ex['income'],
    bins=3,
    labels=['low', 'mid', 'high']
)

# 3. show income + the new columns
print(ex[['income', 'high_earner', 'income_bucket']])

   income  high_earner income_bucket
0   38000        False           low
1   92000         True           mid
2   55000        False           low
3   47000        False           low
4  120000         True          high
5   76000         True           mid
6   88000         True           mid
7   41000        False           low


In [7]:
#5. The preprocessing pipeline

# 🔹 5A. ONE LEAK-FREE PIPELINE: PREPROCESS + MODEL
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_cols = ['age', 'income']
cat_cols = ['city', 'size']

pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipe = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000))])
print(pipe)


# 🔹 5B. FIT THE WHOLE THING IN ONE CALL

X = df[num_cols + cat_cols]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)

pipe.fit(X_train, y_train)
acc = pipe.score(X_test, y_test)
print('Test accuracy:', round(acc, 2))
print('(small toy dataset — the point is the leak-free workflow, not the score)')


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])
Test accuracy: 1.0
(small toy dataset — the point is the leak-free workflow, not the score)


In [25]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

Xn = df[['age', 'income']]
yn = df['bought']

# 1. Pipeline: MinMaxScaler -> LogisticRegression
pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', LogisticRegression())
])

# 2. split + fit on train
X_train, X_test, y_train, y_test = train_test_split(
    Xn, yn, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)

# 3. print test accuracy
print("Test accuracy:", pipe.score(X_test, y_test))

Test accuracy: 0.0
